In [ ]:
# langchain_groq: For integrating with the Groq language model.
# pypdf: To work with PDF files.
# langchain_community: Provides various components for building language model applications.
# langchain: The core LangChain library.
# chromadb: A vector database to store and search text embeddings.
!pip install langchain_groq==1.0.0 pypdf==6.1.1 langchain_community==0.4 langchain==1.0.0 chromadb==1.2.0

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import json
from pypdf import PdfReader

In [ ]:
os.environ["GROQ_API_KEY"] = "API KEY"

In [ ]:
# Initialize Groq LLM
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.7
)

In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"Uploaded File: {file_name}")

In [ ]:
def extract_text(file_path):
    if file_path.endswith(".pdf"):
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
        return text
    elif file_path.endswith(".txt"):
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    else:
        raise ValueError("Unsupported file type. Please upload PDF or TXT.")

text_data = extract_text(file_name)
print("✅ Text extraction complete. Sample:")
print(text_data[:1000])

In [ ]:

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
texts = splitter.split_text(text_data)
print(f"✅ Split text into {len(texts)} chunks.")

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = Chroma.from_texts(texts, embeddings, persist_directory="./vectorstore")
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})
print("✅ Embeddings created and stored in ChromaDB.")


In [ ]:
# Define the expected JSON structure

# parser = JsonOutputParser(...): This line creates an instance of JsonOutputParser. This object is designed to parse the output from the language model and ensure it conforms to a specific JSON structure.
# pydantic_object={...}: This argument specifies the desired JSON structure using a dictionary that mimics a Pydantic model schema.
# "type": "object": Indicates that the expected output is a JSON object.
# "properties": {...}: Defines the expected keys (properties) within the JSON object.
# "patternProperties": {...}: This is used to define the structure for keys that match a regular expression. In this case, it's for keys that are numbers (like "1", "2", "3") and have a specific object structure as their value.
# "correct": {"type": "string"}: Defines the expected structure for the "correct" key within each question object.
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    }
}

parser = JsonOutputParser(pydantic_object={
    "type": "object",
    "patternProperties": {
        "^[0-9]+$": {
            "type": "object",
            "properties": {
                "mcq": {"type": "string"},
                "options": {
                    "type": "object",
                    "properties": {
                        "a": {"type": "string"},
                        "b": {"type": "string"},
                        "c": {"type": "string"},
                        "d": {"type": "string"}
                    },
                    "required": ["a", "b", "c", "d"]
                },
                "correct": {"type": "string"}
            },
            "required": ["mcq", "options", "correct"]
        }
    }
})

In [ ]:
# Create a simple prompt template
TEMPLATE_1 = """
You are an expert MCQ maker. Use the following retrieved text to create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming to the text as well.
Retrieved Text: {context}

Output the response in valid JSON format with double quotes, following the structure of RESPONSE_JSON below. Do not include any other text or markdown outside the JSON block.
Wrap the JSON output within ```json and ``` markers.
### RESPONSE_JSON
{response_json}
"""

prompt = ChatPromptTemplate.from_template(TEMPLATE_1)

In [ ]:
# Create the chain that guarantees JSON output
from operator import itemgetter

chain = (
    {
        "context": itemgetter("text"),
        "number": itemgetter("number"),
        "subject": itemgetter("subject"),
        "tone": itemgetter("tone"),
        "response_json": itemgetter("response_json"),
    }
    | prompt
    | llm
    | parser
)

In [ ]:
# Define the function to generate the quiz
def generate_quiz(text: str, number: int, subject: str, tone: str) -> dict:
    # Use the retriever to get relevant documents based on the text
    # Use the subject as the query for the retriever
    docs = retriever.invoke(subject)
    context = "\n\n".join([doc.page_content for doc in docs])

    result = chain.invoke({"text": context, "number": number, "subject": subject, "tone": tone, "response_json": json.dumps(RESPONSE_JSON, indent=2)})
    print(json.dumps(result, indent=2))

In [ ]:
generate_quiz(text_data, number=5, subject="No Men are foreign", tone="medium")

In [ ]:
!pip show langchain_groq pypdf langchain_community langchain chromadb